# 01 · Dataset exploration

The pipeline supports two data sources:

* **synthetic** (default) — a self-contained generator of paired patches in which every
  scene is rendered through three aligned sensor models: **optical** (true-colour RGB),
  **multispectral** (8-band reflectance stack) and **sar** (single-channel intensity).
* **eurosat** — real EuroSAT RGB patches (multispectral / SAR are derived and flagged `_sim`).

Every patch carries a ground-truth **land-cover class** (10 classes), which is the
semantic relevance used for retrieval evaluation.


In [ ]:
import os, sys
PROJ = os.path.dirname(os.getcwd()) if os.path.split(os.getcwd())[1] == 'notebooks' else os.getcwd()
if PROJ not in sys.path: sys.path.insert(0, PROJ)
nb_dir = os.path.join(PROJ, 'notebooks')
if nb_dir not in sys.path: sys.path.insert(0, nb_dir)
import utils
print('project root:', PROJ)


In [ ]:
# Load the dataset (no model needed for exploration).
import utils
D = utils.load_dataset('configs/default.yaml')
print('source      :', D['cfg']['dataset']['source'])
print('num patches :', len(D['full_ds']))
print('image size  :', D['cfg']['dataset']['image_size'])
print('classes     :', len(D['class_names']))
for m, arr in D['patches'].items():
    print(f'  {m:<14} shape={arr.shape}  dtype={arr.dtype}  range=[{arr.min():.3f},{arr.max():.3f}]')

In [ ]:
utils.class_distribution(D);

### One patch, three sensors
The same geographic scene viewed by optical, multispectral and SAR sensors.

In [ ]:
import numpy as np
for i in [0, 42, 123, 456, 2000]:
    utils.show_all_modalities(D, int(i));

### Grids of a single modality

In [ ]:
utils.show_modalities(D, np.arange(0, 24), modality='optical', cols=6);

In [ ]:
utils.show_modalities(D, np.arange(0, 24), modality='multispectral', cols=6);

In [ ]:
utils.show_modalities(D, np.arange(0, 24), modality='sar', cols=6);

### Spectral signatures
Mean reflectance of each class across the 8 multispectral bands (Blue…SWIR1).
Classes with distinct spectra (e.g. water vs vegetation) are easier to separate.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
ms = D['patches']['multispectral']
bands = D['cfg']['modalities'] and ['Blue','Green','Red','RE1','RE2','NIR1','NIR2','SWIR1']
means = np.stack([ms[D['labels']==c].mean(axis=(0,2,3)) for c in range(len(D['class_names']))])
fig, ax = plt.subplots(figsize=(10,4.5))
for c in range(len(D['class_names'])):
    ax.plot(bands, means[c], marker='o', label=D['class_names'][c])
ax.set_title('Mean per-band reflectance by class'); ax.legend(fontsize=8);
ax.set_xlabel('band'); ax.set_ylabel('reflectance'); plt.xticks(rotation=45)
plt.tight_layout();

---
Next: [02_model_and_embeddings.ipynb](02_model_and_embeddings.ipynb) computes the
shared embedding space with the trained encoder.